# Baseline Teacher YOLO26s — IndoTraffic — Seed 42

Notebook ini digunakan untuk **melatih dan mendiagnosis kandidat teacher YOLO26s** sebelum penelitian Knowledge Distillation (KD) dijalankan.

Tujuan notebook ini **bukan** membuktikan metode KD. Tahap ini hanya menjawab pertanyaan prasyarat:

> **Apakah YOLO26s yang dilatih pada IndoTraffic benar-benar memiliki kemampuan deteksi yang lebih kuat daripada baseline student YOLO26n, terutama pada failure mode yang relevan untuk penelitian?**

## Posisi dalam desain penelitian

- **Teacher candidate:** YOLO26s
- **Student baseline:** YOLO26n
- **Dataset:** IndoTraffic / audited V2
- **Seed teacher:** 42
- **Test set:** tidak digunakan untuk pemilihan teacher atau metode KD
- **Validation:** digunakan untuk teacher viability check dan diagnosis
- **Teacher dilatih satu kali**, kemudian checkpoint yang dibekukan akan digunakan ulang oleh seluruh metode KD.

Baseline student YOLO26n yang menjadi referensi awal:
- Precision: 0.767
- Recall: 0.672
- mAP50: 0.744
- mAP50-95: 0.527
- AP-S dari failure analysis terdahulu: 0.321

Failure mode utama yang sudah ditemukan pada YOLO26n:
- small-object recall relatif rendah;
- small motorcycle lebih sulit dibanding sebagian kelas lain;
- performa memburuk pada luminansi rendah dan overlap tinggi.

Notebook ini mengikuti protokol notebook referensi yang diberikan, tetapi **menghapus seluruh intervensi DySample** dan mengganti model menjadi `yolo26s.pt`.

Referensi resmi YOLO26:
- https://docs.ultralytics.com/models/yolo26/
- https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/models/26/yolo26.yaml


## 1. Instalasi dependensi yang dipin

Versi dependency dipertahankan sama dengan notebook referensi agar perubahan utama hanya berupa **scale model YOLO26n → YOLO26s**.


In [ ]:
%pip install -q "ultralytics==8.4.155" "wandb==0.28.1" "roboflow==1.4.0" pyyaml pandas matplotlib pillow

# Aktifkan integrasi W&B native sebelum import YOLO.
!yolo settings wandb=True


## 2. Environment, GPU, dan direktori


In [ ]:
import gc
import hashlib
import json
import os
import platform
import random
import shutil
import sys
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
import wandb
import yaml
from PIL import Image, ImageOps

IN_KAGGLE = Path("/kaggle/working").exists()
IN_COLAB = "google.colab" in sys.modules
PLATFORM = "kaggle" if IN_KAGGLE else "colab" if IN_COLAB else "local"

if IN_KAGGLE:
    WORK_ROOT = Path("/kaggle/working/yolo26s_indotraffic_teacher_seed42")
elif IN_COLAB:
    WORK_ROOT = Path("/content/yolo26s_indotraffic_teacher_seed42")
else:
    WORK_ROOT = Path("./yolo26s_indotraffic_teacher_seed42")

WORK_ROOT = WORK_ROOT.resolve()
DATA_ROOT = WORK_ROOT / "dataset"
REPORTS_ROOT = WORK_ROOT / "reports"
REPORTS_ROOT.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU tidak tersedia. Aktifkan GPU sebelum training.")

DEVICE = 0
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3

ENV_INFO = {
    "platform": PLATFORM,
    "python": sys.version.split()[0],
    "pytorch": torch.__version__,
    "cuda": torch.version.cuda,
    "ultralytics": ultralytics.__version__,
    "wandb": wandb.__version__,
    "gpu": GPU_NAME,
    "gpu_vram_gb": round(GPU_VRAM_GB, 2),
}

print(json.dumps(ENV_INFO, indent=2))
print("WORK_ROOT:", WORK_ROOT)


## 3. Konfigurasi eksperimen

Hyperparameter training sengaja mengikuti baseline YOLO26n referensi:
- input 640;
- batch 16;
- 100 epoch;
- AdamW;
- LR dan regularisasi identik;
- augmentation identik.

**Catatan Kaggle:** notebook sengaja tidak mengizinkan hidden auto-batch recovery. Jika YOLO26s OOM pada `batch=16`, hentikan run, ubah `CFG["batch"]` secara eksplisit (mis. 8), lalu jalankan fresh run. Teacher hanya dilatih satu kali dan akan dibekukan, sehingga batch teacher boleh berbeda dari student **asal didokumentasikan dan tidak berubah lagi setelah teacher dipilih**.


In [ ]:
CFG = {
    "seed": 42,

    # Dataset
    "workspace": "ilham-rafiqin",
    "project": "poltrada-original-raw-2",
    "dataset_version": 2,
    "class_names": [
        "Bus",
        "Mobil Penumpang",
        "Pejalan Kaki",
        "Sepeda Motor",
        "Truck",
        "Unmotorized",
    ],

    # Teacher architecture
    "pretrained_model": "yolo26s.pt",
    "role": "teacher_candidate",

    # Training — mengikuti baseline reference
    "imgsz": 640,
    "batch": 16,
    "epochs": 100,
    "patience": 25,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "amp": True,
    "workers": 2,
    "cache": False,
    "close_mosaic": 10,
    "flipud": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,

    # Output
    "save_period": -1,
    "plots": True,

    # W&B / Ultralytics output
    "wandb_project": "skripsi-yolo26-kd-indotraffic",
    "wandb_group": "teacher-baseline",
    "run_name": "yolo26s-teacher-seed42-baseline-v1",
}

EXPECTED_SPLITS = {
    "train": {"images": 6775, "labels": 6775, "annotations": 125108, "empty": 16},
    "val":   {"images": 1790, "labels": 1790, "annotations": 32076,  "empty": 2},
    "test":  {"images": 913,  "labels": 913, "annotations": 16158,  "empty": 9},
}

# Reference student baseline — hanya untuk diagnosis teacher.
BASELINE_NATIVE = {
    "Precision": 0.767,
    "Recall": 0.672,
    "mAP50": 0.744,
    "mAP50-95": 0.527,
}

BASELINE_CLASS_MAP5095 = {
    "Bus": 0.668,
    "Mobil Penumpang": 0.720,
    "Pejalan Kaki": 0.364,
    "Sepeda Motor": 0.407,
    "Truck": 0.625,
    "Unmotorized": 0.376,
}

BASELINE_SLICE_RECALL = {
    "All": (32076 - 10500) / 32076,
    "Small": (25393 - 9734) / 25393,
    "Small Sepeda Motor": 0.5606,
    "High overlap": 0.4282,
    "Dark": 0.4713,
    "Small × High overlap": 0.3766,
    "Small × Dark": 0.4005,
    "Small Sepeda Motor × Dark": 0.3094,
    "Small Sepeda Motor × Dark × High overlap": 0.1751,
}

print(json.dumps(CFG, indent=2))


## 4. Secrets dan aktivasi W&B native


In [ ]:
def get_secret(name):
    value = os.getenv(name)
    if value:
        return value

    if IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        except Exception:
            pass

    if IN_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(name)
        except Exception:
            pass

    return None

ROBOFLOW_API_KEY = get_secret("ROBOFLOW_API_KEY")
WANDB_API_KEY = get_secret("WANDB_API_KEY")

if not ROBOFLOW_API_KEY:
    raise RuntimeError("ROBOFLOW_API_KEY tidak ditemukan.")
if not WANDB_API_KEY:
    raise RuntimeError("WANDB_API_KEY tidak ditemukan. Run penelitian wajib tercatat di W&B.")

wandb.login(key=WANDB_API_KEY, relogin=True)

os.environ["WANDB_RUN_GROUP"] = CFG["wandb_group"]
os.environ["WANDB_JOB_TYPE"] = "train"

from ultralytics import YOLO, settings

if settings.get("wandb") is not True:
    raise RuntimeError("Ultralytics W&B integration belum aktif.")

print("Ultralytics W&B setting:", settings.get("wandb"))


## 5. Download exact audited V2

Mengikuti notebook referensi:
- workspace, project, dan version harus identik;
- split harus identik;
- polygon residual dinormalisasi deterministik menjadi enclosing bounding box;
- dataset identity gate harus lolos sebelum training.


In [ ]:
from roboflow import Roboflow

if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
rf_project = rf.workspace(CFG["workspace"]).project(CFG["project"])
downloaded = rf_project.version(CFG["dataset_version"]).download(
    "yolov8",
    location=str(DATA_ROOT),
    overwrite=True,
)

yaml_candidates = []
for root in [DATA_ROOT, Path(downloaded.location)]:
    if root.exists():
        yaml_candidates.extend(root.rglob("data.yaml"))

yaml_candidates = list(dict.fromkeys(p.resolve() for p in yaml_candidates))
if not yaml_candidates:
    raise FileNotFoundError("data.yaml tidak ditemukan setelah download.")

SOURCE_YAML = yaml_candidates[0]
raw_cfg = yaml.safe_load(SOURCE_YAML.read_text(encoding="utf-8"))

names = raw_cfg.get("names", [])
if isinstance(names, dict):
    names = [names[k] for k in sorted(names, key=lambda x: int(x))]
else:
    names = list(names)

if names != CFG["class_names"]:
    raise ValueError(f"Class order berbeda.\nExpected: {CFG['class_names']}\nFound: {names}")

def resolve_split(key, aliases=()):
    value = raw_cfg.get(key)
    candidates = []
    if value:
        p = Path(str(value))
        candidates.append(p if p.is_absolute() else (SOURCE_YAML.parent / p).resolve())
    for alias in aliases:
        candidates.append((SOURCE_YAML.parent / alias / "images").resolve())
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Split {key} tidak ditemukan. Candidates: {candidates}")

SPLIT_IMAGES = {
    "train": resolve_split("train", ("train",)),
    "val": resolve_split("val", ("valid", "val")),
    "test": resolve_split("test", ("test",)),
}

print("SOURCE_YAML:", SOURCE_YAML)
for split, path in SPLIT_IMAGES.items():
    print(f"{split:5s}:", path)


## 6. Normalisasi polygon → enclosing bounding box


In [ ]:
BACKUP_ROOT = REPORTS_ROOT / "labels_before_polygon_to_bbox"
if BACKUP_ROOT.exists():
    shutil.rmtree(BACKUP_ROOT)
BACKUP_ROOT.mkdir(parents=True)

conversion_log = []

def polygon_to_yolo_bbox(parts):
    cls = int(float(parts[0]))
    coords = list(map(float, parts[1:]))
    if len(coords) < 6 or len(coords) % 2 != 0:
        raise ValueError("Polygon harus memiliki >=3 titik.")

    xs = coords[0::2]
    ys = coords[1::2]
    x1, x2 = min(xs), max(xs)
    y1, y2 = min(ys), max(ys)

    x1, y1 = max(0.0, x1), max(0.0, y1)
    x2, y2 = min(1.0, x2), min(1.0, y2)

    if x2 <= x1 or y2 <= y1:
        raise ValueError("Polygon menghasilkan bbox tanpa area.")

    xc = (x1 + x2) / 2
    yc = (y1 + y2) / 2
    w = x2 - x1
    h = y2 - y1
    return cls, xc, yc, w, h

for split, images_dir in SPLIT_IMAGES.items():
    labels_dir = images_dir.parent / "labels"
    split_backup = BACKUP_ROOT / split
    shutil.copytree(labels_dir, split_backup)

    for label_path in sorted(labels_dir.glob("*.txt")):
        raw = label_path.read_text(encoding="utf-8").strip()
        if not raw:
            continue

        output = []
        changed = False

        for line_no, line in enumerate(raw.splitlines(), start=1):
            parts = line.split()

            if len(parts) == 5:
                output.append(line)
                continue

            if len(parts) > 5 and (len(parts) - 1) % 2 == 0:
                cls, xc, yc, w, h = polygon_to_yolo_bbox(parts)
                new_line = f"{cls} {xc:.10f} {yc:.10f} {w:.10f} {h:.10f}"
                output.append(new_line)
                conversion_log.append({
                    "split": split,
                    "label": label_path.name,
                    "line": line_no,
                    "original_tokens": len(parts),
                    "converted": new_line,
                })
                changed = True
                continue

            raise ValueError(
                f"Format anotasi tidak dikenali: {label_path}, line {line_no}: {line}"
            )

        if changed:
            label_path.write_text("\n".join(output) + "\n", encoding="utf-8")

conversion_df = pd.DataFrame(conversion_log)
display(conversion_df.head(20))
print("Polygon annotations converted:", len(conversion_df))

if len(conversion_df):
    conversion_df.to_csv(REPORTS_ROOT / "polygon_to_bbox_conversion.csv", index=False)


## 7. Dataset identity gate


In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

audit_rows = []
fingerprint_hasher = hashlib.sha256()

for split, images_dir in SPLIT_IMAGES.items():
    labels_dir = images_dir.parent / "labels"
    images = sorted(p for p in images_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
    labels = sorted(labels_dir.glob("*.txt"))

    annotations = 0
    empty = 0
    malformed = 0
    polygon_remaining = 0

    for label_path in labels:
        text = label_path.read_text(encoding="utf-8").strip()
        fingerprint_hasher.update(split.encode())
        fingerprint_hasher.update(label_path.name.encode())
        fingerprint_hasher.update(label_path.read_bytes())

        if not text:
            empty += 1
            continue

        for line in text.splitlines():
            parts = line.split()
            annotations += 1

            if len(parts) > 5:
                polygon_remaining += 1
                malformed += 1
                continue

            if len(parts) != 5:
                malformed += 1
                continue

            try:
                cls = int(float(parts[0]))
                xc, yc, w, h = map(float, parts[1:])
            except ValueError:
                malformed += 1
                continue

            if not (0 <= cls < len(names)):
                malformed += 1
            elif not all(np.isfinite([xc, yc, w, h])):
                malformed += 1
            elif w <= 0 or h <= 0:
                malformed += 1

    expected = EXPECTED_SPLITS[split]

    row = {
        "split": split,
        "images": len(images),
        "labels": len(labels),
        "annotations": annotations,
        "empty": empty,
        "malformed_format": malformed,
        "polygon_remaining": polygon_remaining,
    }
    audit_rows.append(row)

    assert len(images) == expected["images"], (split, "images", len(images), expected["images"])
    assert len(labels) == expected["labels"], (split, "labels", len(labels), expected["labels"])
    assert annotations == expected["annotations"], (split, "annotations", annotations, expected["annotations"])
    assert empty == expected["empty"], (split, "empty", empty, expected["empty"])
    assert malformed == 0, (split, "malformed", malformed)
    assert polygon_remaining == 0, (split, "polygon_remaining", polygon_remaining)

AUDIT_DF = pd.DataFrame(audit_rows)
DATASET_FINGERPRINT = fingerprint_hasher.hexdigest()

display(AUDIT_DF)
print("Normalized-label fingerprint:", DATASET_FINGERPRINT)
print("DATASET IDENTITY GATE: PASSED")

AUDIT_DF.to_csv(REPORTS_ROOT / "dataset_identity_gate.csv", index=False)


## 8. Runtime YAML


In [ ]:
runtime_cfg = {
    "path": str(SOURCE_YAML.parent.resolve()),
    "train": str(SPLIT_IMAGES["train"]),
    "val": str(SPLIT_IMAGES["val"]),
    "test": str(SPLIT_IMAGES["test"]),
    "nc": len(names),
    "names": names,
}

RUNTIME_YAML = WORK_ROOT / "data_runtime.yaml"
RUNTIME_YAML.write_text(
    yaml.safe_dump(runtime_cfg, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

print(RUNTIME_YAML.read_text(encoding="utf-8"))


## 9. Architecture audit — YOLO26s

Sebelum training, notebook memverifikasi kandidat teacher yang benar-benar dimuat, jumlah parameter, dan detection head.

Tujuan audit ini:
1. memastikan `yolo26s.pt` berhasil dimuat;
2. merekam kompleksitas parameter aktual pada environment yang dipakai;
3. memastikan model tetap detektor P3/P4/P5 native tanpa modifikasi arsitektur penelitian.


In [ ]:
probe = YOLO(CFG["pretrained_model"])

teacher_total_params = sum(p.numel() for p in probe.model.parameters())
teacher_trainable_params = sum(p.numel() for p in probe.model.parameters() if p.requires_grad)

detect_layers = [
    (i, m.__class__.__name__)
    for i, m in enumerate(probe.model.model)
    if "Detect" in m.__class__.__name__
]

ARCH_AUDIT = {
    "checkpoint": CFG["pretrained_model"],
    "total_params": int(teacher_total_params),
    "trainable_params": int(teacher_trainable_params),
    "detect_layers": detect_layers,
    "model_task": getattr(probe, "task", None),
}

print(json.dumps(ARCH_AUDIT, indent=2))
probe.info(verbose=False)

assert teacher_total_params > 5_000_000, (
    "Parameter teacher terlalu kecil untuk YOLO26s; cek checkpoint/model scale."
)
assert len(detect_layers) == 1, f"Detect head tidak sesuai: {detect_layers}"

del probe
gc.collect()
torch.cuda.empty_cache()
print("ARCHITECTURE GATE: PASSED")


## 10. Reproducibility + fixed-batch safeguards

Run final tidak boleh diam-diam mengganti batch akibat OOM recovery. Jika `batch=16` tidak muat:
1. biarkan notebook gagal;
2. ubah `CFG["batch"]` secara eksplisit;
3. gunakan nama run baru;
4. dokumentasikan perubahan;
5. teacher yang terpilih kemudian dibekukan untuk seluruh eksperimen KD.


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def release_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()

set_seed(CFG["seed"])
release_gpu_memory()
torch.cuda.reset_peak_memory_stats()

RUN_META = {
    "wandb_path": None,
    "wandb_url": None,
    "actual_batch": None,
}

def on_train_start_research(trainer):
    RUN_META["actual_batch"] = int(trainer.batch_size)

    # Model gate: jumlah parameter trainer harus konsisten dengan candidate teacher.
    trained_params = sum(p.numel() for p in trainer.model.parameters())
    if abs(trained_params - teacher_total_params) > 100_000:
        raise RuntimeError(
            f"Parameter model trainer berubah tidak wajar: {trained_params:,} vs probe {teacher_total_params:,}"
        )

    if wandb.run is not None:
        RUN_META["wandb_path"] = wandb.run.path
        RUN_META["wandb_url"] = wandb.run.url
        wandb.run.config.update(
            {
                "research/role": "teacher_candidate",
                "research/architecture": "YOLO26s native",
                "research/student_reference": "YOLO26n",
                "research/seed": CFG["seed"],
                "research/dataset_fingerprint": DATASET_FINGERPRINT,
                "research/dataset_version": CFG["dataset_version"],
                "research/expected_batch": CFG["batch"],
                "research/gpu": GPU_NAME,
                "research/teacher_total_params": teacher_total_params,
            },
            allow_val_change=True,
        )
        print("W&B run:", wandb.run.url)

def enforce_fixed_batch(trainer):
    actual = int(trainer.batch_size)
    RUN_META["actual_batch"] = actual

    if actual != int(CFG["batch"]):
        raise RuntimeError(
            f"Batch berubah dari {CFG['batch']} menjadi {actual}. "
            "Kemungkinan OOM auto-recovery. Hentikan run dan ubah batch secara eksplisit "
            "di CFG agar teacher baseline tetap terdokumentasi."
        )


## 11. Training YOLO26s teacher candidate — seed42

Model dilatih native tanpa custom module dan tanpa distillation.

**Test set belum disentuh.**


In [ ]:
RUN_NAME = CFG["run_name"]

os.chdir(WORK_ROOT)

EXPECTED_RUN_DIR = WORK_ROOT / CFG["wandb_project"] / RUN_NAME
if EXPECTED_RUN_DIR.exists():
    raise RuntimeError(
        f"Run directory sudah ada: {EXPECTED_RUN_DIR}\n"
        "Hapus/rename secara eksplisit jika memang ingin fresh run."
    )

model = YOLO(CFG["pretrained_model"])
model.add_callback("on_train_start", on_train_start_research)
model.add_callback("on_train_epoch_end", enforce_fixed_batch)

started = time.perf_counter()
status = "failed"
caught_error = None

try:
    results = model.train(
        data=str(RUNTIME_YAML),
        epochs=CFG["epochs"],
        imgsz=CFG["imgsz"],
        batch=CFG["batch"],
        device=DEVICE,
        workers=CFG["workers"],
        optimizer=CFG["optimizer"],
        lr0=CFG["lr0"],
        weight_decay=CFG["weight_decay"],
        warmup_epochs=CFG["warmup_epochs"],
        patience=CFG["patience"],
        amp=CFG["amp"],
        seed=CFG["seed"],
        deterministic=True,
        cache=CFG["cache"],
        flipud=CFG["flipud"],
        mixup=CFG["mixup"],
        cutmix=CFG["cutmix"],
        close_mosaic=CFG["close_mosaic"],
        save=True,
        save_period=CFG["save_period"],
        plots=CFG["plots"],
        val=True,
        project=CFG["wandb_project"],
        name=RUN_NAME,
        exist_ok=False,
        verbose=True,
    )
    status = "completed"

except Exception as exc:
    caught_error = exc
    raise

finally:
    elapsed = time.perf_counter() - started
    peak_vram_gb = torch.cuda.max_memory_allocated() / 1024**3

    TRAIN_META = {
        "status": status,
        "seed": CFG["seed"],
        "role": "teacher_candidate",
        "architecture": "YOLO26s native",
        "student_reference": "YOLO26n",
        "dataset_fingerprint": DATASET_FINGERPRINT,
        "duration_seconds": elapsed,
        "duration_hours": elapsed / 3600,
        "peak_vram_gb": peak_vram_gb,
        "requested_batch": CFG["batch"],
        "actual_batch": RUN_META["actual_batch"],
        "wandb_path": RUN_META["wandb_path"],
        "wandb_url": RUN_META["wandb_url"],
        "error": None if caught_error is None else f"{type(caught_error).__name__}: {caught_error}",
        "environment": ENV_INFO,
        "config": CFG,
        "architecture_audit": ARCH_AUDIT,
    }

    (REPORTS_ROOT / "training_metadata.json").write_text(
        json.dumps(TRAIN_META, indent=2),
        encoding="utf-8",
    )

    print(json.dumps(TRAIN_META, indent=2))
    release_gpu_memory()


## 12. Training-output gate


In [ ]:
RUN_DIR = WORK_ROOT / CFG["wandb_project"] / RUN_NAME
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"
RESULTS_CSV = RUN_DIR / "results.csv"
ARGS_YAML = RUN_DIR / "args.yaml"

required_files = [BEST_PT, LAST_PT, RESULTS_CSV, ARGS_YAML]
missing = [str(p) for p in required_files if not p.exists()]

if missing:
    raise RuntimeError(
        "Output training tidak lengkap:\n" + "\n".join(missing)
    )

history = pd.read_csv(RESULTS_CSV)
history.columns = [c.strip() for c in history.columns]

groups = {
    "train_losses": [c for c in history.columns if c.startswith("train/")],
    "val_losses": [c for c in history.columns if c.startswith("val/")],
    "metrics": [c for c in history.columns if c.startswith("metrics/")],
    "learning_rates": [c for c in history.columns if c.startswith("lr/")],
}

for group, cols in groups.items():
    print(f"{group:15s}: {cols}")
    if not cols:
        raise RuntimeError(f"Kelompok metrik '{group}' tidak ditemukan.")

display(history.tail())

map_col = next(
    (c for c in history.columns if "mAP50-95" in c),
    None,
)

if map_col:
    best_idx = history[map_col].astype(float).idxmax()
    print("\nBest epoch row:")
    display(history.loc[[best_idx]])

print("\nTRAINING METRIC GATE: PASSED")
print("best.pt:", BEST_PT)
print("W&B    :", RUN_META.get("wandb_url"))


## 13. Kurva training


In [ ]:
plot_cols = (
    groups["train_losses"]
    + groups["val_losses"]
    + [
        c for c in groups["metrics"]
        if "mAP" in c or "precision" in c or "recall" in c
    ]
)

for col in plot_cols:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(np.arange(1, len(history) + 1), history[col])
    ax.set_xlabel("Epoch")
    ax.set_ylabel(col)
    ax.set_title(col)
    ax.grid(alpha=0.25)
    plt.show()
    plt.close(fig)


## 14. Native validation — teacher vs student baseline

Ini adalah comparison pertama untuk menjawab **apakah YOLO26s layak menjadi teacher**.

Teacher idealnya memiliki:
- mAP50-95 lebih tinggi daripada YOLO26n;
- recall lebih tinggi;
- kemampuan kelas yang bermasalah pada student (terutama Sepeda Motor) yang setidaknya tidak lebih buruk.

Teacher **tidak dimasukkan sebagai kondisi dalam uji statistik student**. Ia berfungsi sebagai fixed reference/source of knowledge.


In [ ]:
release_gpu_memory()
best_model = YOLO(str(BEST_PT))

VAL_PROJECT = WORK_ROOT / "teacher_evaluation"

metrics = best_model.val(
    data=str(RUNTIME_YAML),
    split="val",
    imgsz=CFG["imgsz"],
    batch=CFG["batch"],
    device=DEVICE,
    plots=True,
    project=str(VAL_PROJECT),
    name="native_validation",
    exist_ok=True,
    verbose=True,
)

native_summary = {
    "Precision": float(metrics.box.mp),
    "Recall": float(metrics.box.mr),
    "mAP50": float(metrics.box.map50),
    "mAP50-95": float(metrics.box.map),
}

native_compare = pd.DataFrame({
    "Metric": list(native_summary.keys()),
    "YOLO26n_baseline_seed42": [BASELINE_NATIVE[k] for k in native_summary],
    "YOLO26s_teacher_seed42": [native_summary[k] for k in native_summary],
})

native_compare["Delta_teacher_minus_student"] = (
    native_compare["YOLO26s_teacher_seed42"]
    - native_compare["YOLO26n_baseline_seed42"]
)

display(native_compare.style.format({
    "YOLO26n_baseline_seed42": "{:.4f}",
    "YOLO26s_teacher_seed42": "{:.4f}",
    "Delta_teacher_minus_student": "{:+.4f}",
}))

native_compare.to_csv(
    REPORTS_ROOT / "teacher_vs_student_native_validation.csv",
    index=False,
)

teacher_overall_headroom = (
    native_summary["mAP50-95"] - BASELINE_NATIVE["mAP50-95"]
)

print(f"Teacher mAP50-95 headroom: {teacher_overall_headroom:+.4f}")
if teacher_overall_headroom <= 0:
    print(
        "WARNING: YOLO26s tidak mengungguli student pada mAP50-95. "
        "Jangan lanjut ke KD sebelum penyebabnya diaudit."
    )
else:
    print("OVERALL TEACHER HEADROOM: POSITIVE")


## 15. Per-class validation comparison


In [ ]:
teacher_maps = np.asarray(metrics.box.maps, dtype=float)

if len(teacher_maps) != len(CFG["class_names"]):
    raise RuntimeError(
        f"Jumlah class AP tidak sesuai: {len(teacher_maps)} vs {len(CFG['class_names'])}"
    )

class_compare = pd.DataFrame({
    "Class": CFG["class_names"],
    "YOLO26n_baseline_mAP50-95": [
        BASELINE_CLASS_MAP5095[name] for name in CFG["class_names"]
    ],
    "YOLO26s_teacher_mAP50-95": teacher_maps,
})

class_compare["Delta"] = (
    class_compare["YOLO26s_teacher_mAP50-95"]
    - class_compare["YOLO26n_baseline_mAP50-95"]
)

display(
    class_compare.style.format({
        "YOLO26n_baseline_mAP50-95": "{:.4f}",
        "YOLO26s_teacher_mAP50-95": "{:.4f}",
        "Delta": "{:+.4f}",
    })
)

class_compare.to_csv(
    REPORTS_ROOT / "teacher_vs_student_class_map5095.csv",
    index=False,
)

motor_delta = float(
    class_compare.loc[
        class_compare["Class"].eq("Sepeda Motor"),
        "Delta"
    ].iloc[0]
)

print(f"Sepeda Motor mAP50-95 delta: {motor_delta:+.4f}")


## 16. Failure-oriented teacher diagnostic

Untuk menjaga kesinambungan dengan failure analysis YOLO26n, teacher diperiksa pada operating point diagnostik yang sama:

- confidence = **0.388**
- IoU = **0.50**
- matching = same-class greedy matching

Ini **bukan** pengganti mAP resmi. Bagian ini hanya menjawab:

> Apakah teacher memiliki headroom pada slice yang sebelumnya sulit bagi student?


In [ ]:
VAL_IMAGES = SPLIT_IMAGES["val"]
VAL_LABELS = VAL_IMAGES.parent / "labels"

CONF_THRESHOLD = 0.388
IOU_THRESHOLD = 0.50
SMALL_MAX = 32 ** 2
MEDIUM_MAX = 96 ** 2

def yolo_to_xyxy(xc, yc, bw, bh, width, height):
    xc *= width
    yc *= height
    bw *= width
    bh *= height
    return np.array([
        xc - bw / 2,
        yc - bh / 2,
        xc + bw / 2,
        yc + bh / 2,
    ], dtype=float)

def iou_xyxy(a, b):
    xa = max(a[0], b[0])
    ya = max(a[1], b[1])
    xb = min(a[2], b[2])
    yb = min(a[3], b[3])

    iw = max(0.0, xb - xa)
    ih = max(0.0, yb - ya)
    inter = iw * ih

    area_a = max(0.0, a[2]-a[0]) * max(0.0, a[3]-a[1])
    area_b = max(0.0, b[2]-b[0]) * max(0.0, b[3]-b[1])

    union = area_a + area_b - inter
    return 0.0 if union <= 0 else inter / union

def target_overlap_ratio(target, other):
    xa = max(target[0], other[0])
    ya = max(target[1], other[1])
    xb = min(target[2], other[2])
    yb = min(target[3], other[3])

    iw = max(0.0, xb - xa)
    ih = max(0.0, yb - ya)
    inter = iw * ih

    target_area = (
        max(0.0, target[2]-target[0])
        * max(0.0, target[3]-target[1])
    )
    return 0.0 if target_area <= 0 else inter / target_area

def scale_from_area(area_640):
    if area_640 < SMALL_MAX:
        return "Small"
    if area_640 < MEDIUM_MAX:
        return "Medium"
    return "Large"


In [ ]:
# ------------------------------------------------------------
# Build GT metadata + brightness + overlap
# ------------------------------------------------------------

gt_by_file = {}
image_brightness = {}
all_objects = []

image_paths = sorted(
    p for p in VAL_IMAGES.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
)

for image_path in image_paths:
    with Image.open(image_path) as im:
        width, height = im.size
        gray = ImageOps.grayscale(im)
        brightness = float(np.asarray(gray, dtype=np.float32).mean())

    image_brightness[image_path.name] = brightness

    label_path = VAL_LABELS / f"{image_path.stem}.txt"
    objects = []

    text = label_path.read_text(encoding="utf-8").strip()
    if text:
        for gt_idx, line in enumerate(text.splitlines()):
            cls, xc, yc, bw, bh = map(float, line.split())
            cls = int(cls)

            xyxy = yolo_to_xyxy(
                xc, yc, bw, bh,
                width, height,
            )

            gain = min(CFG["imgsz"] / width, CFG["imgsz"] / height)
            bw_640 = bw * width * gain
            bh_640 = bh * height * gain
            area_640 = bw_640 * bh_640

            obj = {
                "file_name": image_path.name,
                "gt_idx": gt_idx,
                "class_id": cls,
                "Class": CFG["class_names"][cls],
                "bbox": xyxy,
                "Scale": scale_from_area(area_640),
                "brightness": brightness,
            }
            objects.append(obj)

    # max target-overlap terhadap GT lain pada image yang sama
    for i, obj in enumerate(objects):
        ratios = [
            target_overlap_ratio(obj["bbox"], other["bbox"])
            for j, other in enumerate(objects)
            if j != i
        ]
        obj["max_overlap"] = max(ratios, default=0.0)
        all_objects.append(obj)

    gt_by_file[image_path.name] = objects

brightness_values = np.array(list(image_brightness.values()), dtype=float)
q33_brightness = float(np.quantile(brightness_values, 0.33))
q67_brightness = float(np.quantile(brightness_values, 0.67))

positive_overlaps = np.array(
    [o["max_overlap"] for o in all_objects if o["max_overlap"] > 0],
    dtype=float,
)
q33_overlap = float(np.quantile(positive_overlaps, 0.33))
q67_overlap = float(np.quantile(positive_overlaps, 0.67))

def brightness_group(v):
    if v <= q33_brightness:
        return "Dark"
    if v <= q67_brightness:
        return "Medium"
    return "Bright"

def overlap_group(v):
    if v == 0:
        return "None"
    if v <= q33_overlap:
        return "Low"
    if v <= q67_overlap:
        return "Medium"
    return "High"

for obj in all_objects:
    obj["Brightness_Group"] = brightness_group(obj["brightness"])
    obj["Overlap"] = overlap_group(obj["max_overlap"])

print("Brightness thresholds:", q33_brightness, q67_brightness)
print("Overlap thresholds:", q33_overlap, q67_overlap)
print("GT objects:", len(all_objects))

assert len(all_objects) == EXPECTED_SPLITS["val"]["annotations"]


In [ ]:
# ------------------------------------------------------------
# Prediction cache — stream agar hemat RAM
# ------------------------------------------------------------

prediction_cache = {}

pred_stream = best_model.predict(
    source=str(VAL_IMAGES),
    imgsz=CFG["imgsz"],
    conf=0.001,
    max_det=300,
    device=DEVICE,
    batch=1,
    stream=True,
    verbose=False,
)

for result in pred_stream:
    file_name = Path(result.path).name

    if result.boxes is None or len(result.boxes) == 0:
        prediction_cache[file_name] = []
        continue

    xyxy = result.boxes.xyxy.detach().cpu().numpy()
    cls = result.boxes.cls.detach().cpu().numpy().astype(int)
    conf = result.boxes.conf.detach().cpu().numpy()

    prediction_cache[file_name] = [
        {
            "bbox": xyxy[i].astype(float),
            "class_id": int(cls[i]),
            "score": float(conf[i]),
        }
        for i in range(len(conf))
    ]

assert len(prediction_cache) == EXPECTED_SPLITS["val"]["images"]
print("Prediction cache images:", len(prediction_cache))


In [ ]:
# ------------------------------------------------------------
# Same-class greedy matching @ fixed operating point
# ------------------------------------------------------------

records = []

for file_name, gt_list in gt_by_file.items():
    preds = [
        p for p in prediction_cache.get(file_name, [])
        if p["score"] >= CONF_THRESHOLD
    ]
    preds = sorted(preds, key=lambda x: x["score"], reverse=True)

    matched_gt = set()

    for pred in preds:
        best_iou = 0.0
        best_idx = None

        for gt_idx, gt in enumerate(gt_list):
            if gt_idx in matched_gt:
                continue
            if pred["class_id"] != gt["class_id"]:
                continue

            iou = iou_xyxy(pred["bbox"], gt["bbox"])
            if iou > best_iou:
                best_iou = iou
                best_idx = gt_idx

        if best_idx is not None and best_iou >= IOU_THRESHOLD:
            matched_gt.add(best_idx)

    for gt_idx, gt in enumerate(gt_list):
        row = {
            **{k: v for k, v in gt.items() if k != "bbox"},
            "Status": "TP" if gt_idx in matched_gt else "FN",
        }
        records.append(row)

teacher_object_df = pd.DataFrame(records)

assert len(teacher_object_df) == EXPECTED_SPLITS["val"]["annotations"]

def recall_slice(name, mask):
    subset = teacher_object_df.loc[mask]
    gt = len(subset)
    tp = int((subset["Status"] == "TP").sum())
    fn = int((subset["Status"] == "FN").sum())
    recall = tp / gt if gt else np.nan
    return {
        "Slice": name,
        "GT": gt,
        "TP": tp,
        "FN": fn,
        "Teacher_Recall": recall,
    }

is_small = teacher_object_df["Scale"].eq("Small")
is_motor = teacher_object_df["Class"].eq("Sepeda Motor")
is_dark = teacher_object_df["Brightness_Group"].eq("Dark")
is_high_overlap = teacher_object_df["Overlap"].eq("High")

slice_rows = [
    recall_slice("All", pd.Series(True, index=teacher_object_df.index)),
    recall_slice("Small", is_small),
    recall_slice("Small Sepeda Motor", is_small & is_motor),
    recall_slice("High overlap", is_high_overlap),
    recall_slice("Dark", is_dark),
    recall_slice("Small × High overlap", is_small & is_high_overlap),
    recall_slice("Small × Dark", is_small & is_dark),
    recall_slice("Small Sepeda Motor × Dark", is_small & is_motor & is_dark),
    recall_slice(
        "Small Sepeda Motor × Dark × High overlap",
        is_small & is_motor & is_dark & is_high_overlap,
    ),
]

slice_df = pd.DataFrame(slice_rows)

slice_df["YOLO26n_Baseline_Recall"] = (
    slice_df["Slice"].map(BASELINE_SLICE_RECALL)
)
slice_df["Delta_teacher_minus_student"] = (
    slice_df["Teacher_Recall"] - slice_df["YOLO26n_Baseline_Recall"]
)

display(
    slice_df.style.format({
        "Teacher_Recall": "{:.4f}",
        "YOLO26n_Baseline_Recall": "{:.4f}",
        "Delta_teacher_minus_student": "{:+.4f}",
    })
)

slice_df.to_csv(
    REPORTS_ROOT / "teacher_vs_student_failure_slices.csv",
    index=False,
)

teacher_object_df.drop(
    columns=["brightness"],
    errors="ignore",
).to_csv(
    REPORTS_ROOT / "teacher_object_level_status.csv",
    index=False,
)


## 17. Teacher viability gate

Teacher tidak otomatis dianggap layak hanya karena lebih besar.

Interpretasi yang dicari:

1. **Wajib:** `mAP50-95(YOLO26s) > mAP50-95(YOLO26n)`.
2. **Diinginkan:** recall overall teacher lebih tinggi.
3. **Sangat relevan untuk penelitian:** teacher memiliki headroom pada `Small`.
4. **Sangat relevan:** teacher lebih baik pada `Small Sepeda Motor`.
5. Kondisi sulit `Dark` / `High overlap` digunakan sebagai diagnosis tambahan.
6. Jika teacher hanya unggul overall tetapi tidak pada target failure mode, KD masih mungkin diteliti, tetapi rasional small-object transfer menjadi lebih lemah.

Tidak ada assert otomatis pada slice karena diagnosis perlu dibaca sebagai pola, bukan satu threshold arbitrer.


In [ ]:
slice_lookup = slice_df.set_index("Slice")

TEACHER_VIABILITY = {
    "overall_map5095_positive": bool(
        native_summary["mAP50-95"] > BASELINE_NATIVE["mAP50-95"]
    ),
    "overall_recall_positive": bool(
        native_summary["Recall"] > BASELINE_NATIVE["Recall"]
    ),
    "small_recall_positive": bool(
        slice_lookup.loc["Small", "Delta_teacher_minus_student"] > 0
    ),
    "small_motorcycle_recall_positive": bool(
        slice_lookup.loc[
            "Small Sepeda Motor",
            "Delta_teacher_minus_student"
        ] > 0
    ),
    "motorcycle_class_map_positive": bool(motor_delta > 0),
}

print(json.dumps(TEACHER_VIABILITY, indent=2))

decision = (
    "LAYAK DILANJUTKAN KE PILOT KD"
    if TEACHER_VIABILITY["overall_map5095_positive"]
    else "STOP / AUDIT TEACHER SEBELUM KD"
)

print("\nTEACHER DECISION:", decision)

(REPORTS_ROOT / "teacher_viability.json").write_text(
    json.dumps(
        {
            "decision": decision,
            "criteria": TEACHER_VIABILITY,
            "native_summary": native_summary,
            "baseline_native": BASELINE_NATIVE,
        },
        indent=2,
    ),
    encoding="utf-8",
)


## 18. Optional — verifikasi remote W&B


In [ ]:
if RUN_META.get("wandb_path"):
    try:
        api = wandb.Api(timeout=60)
        remote_run = api.run(RUN_META["wandb_path"])
        sample = remote_run.history(samples=5, pandas=True)
        print("Remote W&B run:", remote_run.url)
        print("Contoh keys remote:")
        print(sorted(sample.columns.tolist()))
    except Exception as exc:
        print("Remote W&B verification warning:", repr(exc))
else:
    print("W&B path tidak tersedia; cek log training.")


## 19. Paket artefak teacher

Bundle menyimpan checkpoint teacher dan seluruh audit/diagnostic report yang diperlukan untuk tahap KD berikutnya.


In [ ]:
BUNDLE_DIR = WORK_ROOT / "teacher_bundle_yolo26s_seed42"

if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)

BUNDLE_DIR.mkdir(parents=True)

def copy_if_exists(src, dst_name=None):
    src = Path(src)
    if src.exists():
        dst = BUNDLE_DIR / (dst_name or src.name)
        if src.is_file():
            shutil.copy2(src, dst)

for src in [
    BEST_PT,
    LAST_PT,
    RESULTS_CSV,
    ARGS_YAML,
]:
    copy_if_exists(src)

for src in RUN_DIR.glob("*.png"):
    copy_if_exists(src)

for src in REPORTS_ROOT.glob("*"):
    if src.is_file():
        copy_if_exists(src, f"report_{src.name}")

manifest = []

for p in sorted(BUNDLE_DIR.glob("*")):
    if p.is_file():
        manifest.append({
            "file": p.name,
            "bytes": p.stat().st_size,
            "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
        })

(BUNDLE_DIR / "MANIFEST.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

ZIP_PATH = (
    WORK_ROOT
    / "yolo26s_teacher_seed42_baseline_bundle.zip"
)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED,
) as zf:
    for p in sorted(BUNDLE_DIR.glob("*")):
        if p.is_file():
            zf.write(p, arcname=p.name)

print("Bundle :", ZIP_PATH)
print(
    "Size MB:",
    round(
        ZIP_PATH.stat().st_size / 1024**2,
        2,
    ),
)


## 20. Final gate

Notebook selesai apabila:
- training completed;
- `best.pt`, `last.pt`, dan `results.csv` tersedia;
- batch aktual sama dengan batch yang diminta;
- native validation selesai;
- teacher viability report tersimpan.

**Test set tetap belum digunakan.**


In [ ]:
assert status == "completed"
assert BEST_PT.exists()
assert LAST_PT.exists()
assert RESULTS_CSV.exists()
assert RUN_META["actual_batch"] == CFG["batch"]
assert (REPORTS_ROOT / "teacher_viability.json").exists()

print("=" * 76)
print("YOLO26s TEACHER BASELINE — SEED42: COMPLETED")
print("Run dir :", RUN_DIR)
print("best.pt :", BEST_PT)
print("W&B URL :", RUN_META.get("wandb_url"))
print("Bundle  :", ZIP_PATH)
print("=" * 76)

print("\nTeacher vs student — native metrics:")
display(native_compare)

print("\nTeacher vs student — per-class mAP50-95:")
display(class_compare)

print("\nTeacher vs student — failure slices:")
display(slice_df)

print("\nTeacher decision:")
print(decision)
